# 🏺 Hieroglyph NLP Pipeline v6
### AnushS Gardiner→English → spaCy → Seed-X Arabic → Sentiment
---
> **GPU: Tesla T4 (16GB)**  
> الداتا الوحيدة: `intention_dataset.csv`

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'⚠️  {result.stderr[-300:]}')

pip('numpy==1.26.4')
pip('protobuf==3.20.3')
pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
pip('flask', 'flask-cors', 'pyngrok')

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')
print('⚠️  Restart the kernel now (Run → Restart & Run All)')

## Cell 1 — Paths

In [1]:
import os

INTENTION_PATH = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv'

status = '✅' if os.path.exists(INTENTION_PATH) else '❌ NOT FOUND'
print(f'  {status}  Intention: {INTENTION_PATH}')

  ✅  Intention: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv


## Cell 2 — Load Intention Dataset

In [2]:
import pandas as pd

df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip()
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }

print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')

✅ Intention dataset loaded: 139 intentions


## Cell 3 — Load AnushS Hieroglyph Translator
> بياخد Gardiner codes مباشرة → English keywords/meaning

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

HIERO_MODEL = 'AnushS/Hieroglyph-Translator-Using-Gardiner-Codes'
print(f'\n🔄 Loading {HIERO_MODEL}...')
hiero_tokenizer = AutoTokenizer.from_pretrained(HIERO_MODEL)
hiero_model     = AutoModelForSeq2SeqLM.from_pretrained(HIERO_MODEL)
hiero_model     = hiero_model.to(DEVICE).eval()
print(f'✅ Hieroglyph Translator loaded')


def translate_gardiner(gardiner_codes_str):
    """Gardiner codes string → English keywords via AnushS model"""
    inputs = hiero_tokenizer(
        gardiner_codes_str,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to(DEVICE)
    with torch.no_grad():
        outputs = hiero_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )
    return hiero_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


# ── Quick test ──
test = translate_gardiner('D21 N35 N5')
print(f'\n🧪 Test "D21 N35 N5" → "{test}"')

🖥️  Device: cuda
   GPU : Tesla T4
   VRAM: 15.6 GB

🔄 Loading AnushS/Hieroglyph-Translator-Using-Gardiner-Codes...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Hieroglyph Translator loaded

🧪 Test "D21 N35 N5" → "early morning"


## Cell 4 — Load spaCy + Sentiment Models

In [4]:
import numpy as np
import spacy
from transformers import AutoTokenizer as ATok2, AutoModelForSequenceClassification

# ── spaCy ──────────────────────────────────────────
nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')

# ── Sentiment على CPU ───────────────────────────────
print('🔄 Loading sentiment model on CPU...')
SENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer  = ATok2.from_pretrained(SENT_MODEL_NAME)
sent_model      = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
sent_model      = sent_model.to('cpu').eval()
print('✅ Sentiment model loaded (CPU)')

✅ spaCy loaded
🔄 Loading sentiment model on CPU...


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

✅ Sentiment model loaded (CPU)


## Cell 5 — Load Seed-X PPO 7B

In [5]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
from safetensors import safe_open
import transformers.modeling_utils as _mu

# ── Monkey-patch: إصلاح safetensors metadata bug على Kaggle ──
_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass
_mu.safe_open = _PatchedSafeOpen

SEED_X_MODEL = 'ByteDance-Seed/Seed-X-PPO-7B'
print(f'🔄 Loading {SEED_X_MODEL} ...')
print('   float16 — T4 لا تدعم bfloat16')
print('   أول مرة: ~2 دقيقة — من الـ cache: ثواني')

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    SEED_X_MODEL,
    trust_remote_code=True,
)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    SEED_X_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
seed_x_model.eval()
print(f'✅ {SEED_X_MODEL} loaded and ready!')

🔄 Loading ByteDance-Seed/Seed-X-PPO-7B ...
   float16 — T4 لا تدعم bfloat16
   أول مرة: ~2 دقيقة — من الـ cache: ثواني


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ ByteDance-Seed/Seed-X-PPO-7B loaded and ready!


## Cell 6 — NLP Functions

In [6]:
import re

# ──────────────────────────────────────────────────
# 1. spaCy: keywords → proper sentence
# ──────────────────────────────────────────────────
def fix_with_spacy(raw_english):
    cleaned = re.sub(r'\s+', ' ', raw_english).strip()
    doc     = nlp_spacy(cleaned)

    tokens_info = [
        {'text': t.text, 'pos': t.pos_, 'dep': t.dep_, 'lemma': t.lemma_}
        for t in doc if not t.is_punct and not t.is_space
    ]

    has_verb = any(t['pos'] in ('VERB', 'AUX') for t in tokens_info)
    nouns    = [t['text'] for t in tokens_info if t['pos'] in ('NOUN', 'PROPN')]
    adjs     = [t['text'] for t in tokens_info if t['pos'] == 'ADJ']

    if len(tokens_info) == 1:
        fixed = tokens_info[0]['text'].capitalize()
    elif not has_verb and nouns and adjs:
        fixed = f'The {nouns[0]} is {" and ".join(adjs)}'
    elif not has_verb and len(nouns) >= 2:
        fixed = f'The {nouns[0]} of {" ".join(nouns[1:])}'
    elif not has_verb and nouns:
        fixed = f'The {nouns[0]}'
    else:
        fixed = cleaned

    fixed = fixed.strip().capitalize()
    if fixed and fixed[-1] not in '.!?':
        fixed += '.'

    return fixed, tokens_info


# ──────────────────────────────────────────────────
# 2. Seed-X: English → Arabic
# ──────────────────────────────────────────────────
def translate_to_arabic(english_text):
    if not english_text or english_text.startswith('['):
        return ''

    prompt = f'Translate the following English sentence into Arabic:\n{english_text} <ar>'

    try:
        inputs = seed_x_tokenizer(prompt, return_tensors='pt')
        inputs.pop('token_type_ids', None)
        inputs = {k: v.to(seed_x_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seed_x_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=seed_x_tokenizer.eos_token_id,
            )
        arabic = seed_x_tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip()
        return arabic if arabic else english_text
    except Exception as e:
        print(f'❌ Translation error: {str(e)[:100]}')
        return english_text


# ──────────────────────────────────────────────────
# 3. Sentiment
# ──────────────────────────────────────────────────
def analyze_sentiment(text):
    if not text or text.startswith('['):
        return 'neutral', 0.5
    try:
        inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            scores = torch.softmax(sent_model(**inputs).logits, dim=1).numpy()[0]
        idx    = int(np.argmax(scores))
        labels = ['negative', 'neutral', 'positive']
        return labels[idx], round(float(scores[idx]), 3)
    except:
        return 'neutral', 0.5


# ──────────────────────────────────────────────────
# 4. Intention
# ──────────────────────────────────────────────────
def detect_intention(text):
    words  = set(text.lower().split())
    scores = {i: len(words & d['keywords']) for i, d in INTENTION_MAP.items() if words & d['keywords']}
    if not scores:
        return 'descriptive', 'وصفي'
    best = max(scores, key=scores.get)
    return best, INTENTION_MAP[best]['arabic']


print('✅ All NLP functions ready')

✅ All NLP functions ready


## Cell 7 — Master Pipeline

In [ ]:
import time

def run_nlp_pipeline(gardiner_codes_str, verbose=True):
    """
    Pipeline كامل:
    Gardiner codes → AnushS → spaCy → Seed-X → Sentiment + Intention
    """
    t0 = time.time()

    # Step 1: AnushS — Gardiner → raw English keywords
    raw_english = translate_gardiner(gardiner_codes_str)

    # Step 2: spaCy — keywords → proper sentence
    fixed_english, tokens = fix_with_spacy(raw_english)

    # Step 3: Seed-X — English → Arabic
    arabic = translate_to_arabic(fixed_english)

    # Step 4: Sentiment
    sentiment, sent_score = analyze_sentiment(fixed_english)

    # Step 5: Intention
    intention_en, intention_ar = detect_intention(fixed_english)

    result = {
        'input'        : gardiner_codes_str,
        'raw_english'  : raw_english,
        'english'      : fixed_english,
        'arabic'       : arabic,
        'sentiment'    : sentiment,
        'sent_score'   : sent_score,
        'intention_en' : intention_en,
        'intention_ar' : intention_ar,
        'tokens'       : tokens,
        'time'         : round(time.time() - t0, 2),
    }
    (w)sꞽr wnꞽs m n =k ꞽr.t-ḥr.w ꞽꜥb n =k s(ꞽ) ꞽr rʾ =k

    if verbose:
        emoji = '😊' if sentiment == 'positive' else '😞' if sentiment == 'negative' else '😐'
        print('=' * 58)
        print(f'  🏺  HIEROGLYPH NLP PIPELINE')
        print('=' * 58)
        print(f'  Input       : {gardiner_codes_str}')
        print(f'  Raw English : {raw_english}')
        print(f'  English     : {fixed_english}')
        print(f'  Arabic      : {arabic}')
        print(f'  Sentiment   : {emoji} {sentiment} ({sent_score})')
        print(f'  Intention   : {intention_en} — {intention_ar}')
        print(f'  Time        : {result["time"]}s')
        print('=' * 58)

    return result


# ── Test ──
print('🧪 Testing pipeline...')
test = run_nlp_pipeline('D21 N35 N5')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🧪 Testing pipeline...


2026-03-21 01:24:32.640217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774056273.100377     118 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774056273.216188     118 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774056274.256136     118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774056274.256160     118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774056274.256163     118 computation_placer.cc:177] computation placer alr

  🏺  HIEROGLYPH NLP PIPELINE
  Input       : D21 N35 N5
  Raw English : early morning
  English     : The morning is early.
  Arabic      : الصباح مبكر.
  Sentiment   : 😊 positive (0.602)
  Intention   : daily_offering — القربان اليومي
  Time        : 40.41s


## Cell 8 — Flask API + Ngrok

In [8]:
test = run_nlp_pipeline('G17 N35 D21')
print()
test = run_nlp_pipeline('D21 N35 N5')
print()
test = run_nlp_pipeline('D21 N35')
print()
test = run_nlp_pipeline('N5')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  🏺  HIEROGLYPH NLP PIPELINE
  Input       : G17 N35 D21
  Raw English : sceptre
  English     : Sceptre.
  Arabic      : عصا الملوك.
  Sentiment   : 😐 neutral (0.69)
  Intention   : descriptive — وصفي
  Time        : 0.81s



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  🏺  HIEROGLYPH NLP PIPELINE
  Input       : D21 N35 N5
  Raw English : early morning
  English     : The morning is early.
  Arabic      : الصباح مبكر.
  Sentiment   : 😊 positive (0.602)
  Intention   : daily_offering — القربان اليومي
  Time        : 0.68s



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  🏺  HIEROGLYPH NLP PIPELINE
  Input       : D21 N35
  Raw English : the sun
  English     : The sun.
  Arabic      : الشمس.
  Sentiment   : 😊 positive (0.508)
  Intention   : descriptive — وصفي
  Time        : 0.48s

  🏺  HIEROGLYPH NLP PIPELINE
  Input       : N5
  Raw English : N5
  English     : N5.
  Arabic      : N5.
  Sentiment   : 😐 neutral (0.566)
  Intention   : descriptive — وصفي
  Time        : 0.36s


In [ ]:
import os
os.system("fuser -k 5000/tcp")

In [9]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

app = Flask(__name__)
CORS(app)

@app.route('/api/translate', methods=['POST'])
def api_translate():
    data     = request.get_json() or {}
    gardiner = data.get('gardiner', '').strip()
    if not gardiner:
        return jsonify({'error': 'gardiner field is required'}), 400
    try:
        result = run_nlp_pipeline(gardiner, verbose=False)
        return jsonify({'success': True, **result})
    except Exception as e:
        import traceback
        return jsonify({'success': False, 'error': str(e), 'trace': traceback.format_exc()}), 500

@app.route('/api/decipher', methods=['POST'])
def decipher():
    data  = request.get_json() or {}
    codes = data.get('codes', [])
    if not codes:
        return jsonify({'error': 'Missing codes'}), 400
    gardiner = ' '.join(codes)
    try:
        result = run_nlp_pipeline(gardiner, verbose=False)
        return jsonify({'success': True, 'data': result})
    except Exception as e:
        import traceback
        return jsonify({'success': False, 'error': str(e), 'trace': traceback.format_exc()}), 500

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({
        'status' : 'ok',
        'version': 'v6',
        'models' : [
            'AnushS/Hieroglyph-Translator-Using-Gardiner-Codes',
            'ByteDance-Seed/Seed-X-PPO-7B',
            'cardiffnlp/twitter-roberta-base-sentiment',
            'spacy:en_core_web_sm',
        ]
    })

@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify([
        {'name': 'Name of Ra',       'codes': 'D21 N35 N5'},
        {'name': 'Beautiful house',  'codes': 'F35 O1'},
        {'name': 'Offering formula', 'codes': 'R4 Q1 N5'},
        {'name': 'Seated man',       'codes': 'A1'},
        {'name': 'King of Egypt',    'codes': 'L2 N17 N18'},
    ])

NGROK_TOKEN = '3BBILR8WLOgh6uQGi4jU0aN6YoR_68QjQ4J4eRxCyhsjHLWMr'
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f'🌍 Public URL: {public_url}')
    print(f'   POST {public_url}/api/translate')
    print(f'   POST {public_url}/api/decipher')
else:
    print('⚠️  No ngrok token — localhost:5000 only')

def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()
print('✅ Flask API running on port 5000')

🌍 Public URL: https://irretrievably-unsimpering-darrin.ngrok-free.dev                              
   POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/translate
   POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/decipher
✅ Flask API running on port 5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [21/Mar/2026 01:25:48] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 01:25:48] "GET /api/examples HTTP/1.1" 200 -
